<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# ADA Algorithm for CICIDS-2017 Dataset
This notebook provides a robust replication of the `ADA_final.py` script, ensuring all preprocessing steps from the paper are included and performing XAI analysis (SHAP) with 100 samples.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from collections import Counter
from sklearn.preprocessing import label_binarize
import time
import matplotlib.pyplot as plt
import shap
import os

np.random.seed(0)

## Defining Features of Interest
We use the 15 features identified as most significant in the research.

In [ ]:
req_cols = [
    ' Packet Length Std', ' Total Length of Bwd Packets', ' Subflow Bwd Bytes',
    ' Destination Port', ' Packet Length Variance', ' Bwd Packet Length Mean',
    ' Avg Bwd Segment Size', 'Bwd Packet Length Max', ' Init_Win_bytes_backward',
    'Total Length of Fwd Packets', ' Subflow Fwd Bytes', 'Init_Win_bytes_forward',
    ' Average Packet Size', ' Packet Length Mean', ' Max Packet Length', ' Label'
]

## Loading Database

In [ ]:
db_path = '../cicids_db/'
files = [
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
]

frames = []
for f in files:
    path = os.path.join(db_path, f)
    print(f'Loading {f}...')
    df_tmp = pd.read_csv(path, usecols=req_cols, low_memory=False, encoding='cp1252')
    frames.append(df_tmp)

df = pd.concat(frames, ignore_index=True)
df = df.sample(frac=1, random_state=0)
print("Finished loading all databases.")

## Preprocessing (Normalization and Label Grouping)
We clean the column names, handle label replacements, and perform max normalization.

In [ ]:
# Normalize column names (remove leading spaces)
df.columns = df.columns.str.strip()

# Filter out header rows if they appear in concat
df = df[df['Label'] != 'Label']

# Label replacement
y = df['Label'].replace({
    'DoS GoldenEye': 'Dos/Ddos', 'DoS Hulk': 'Dos/Ddos', 'DoS Slowhttptest': 'Dos/Ddos',
    'DoS slowloris': 'Dos/Ddos', 'Heartbleed': 'Dos/Ddos', 'DDoS': 'Dos/Ddos',
    'FTP-Patator': 'Brute Force', 'SSH-Patator': 'Brute Force',
    'Web Attack - Brute Force': 'Web Attack', 'Web Attack - Sql Injection': 'Web Attack', 'Web Attack - XSS': 'Web Attack',
    'Web Attack XSS': 'Web Attack', 'Web Attack Sql Injection': 'Web Attack', 'Web Attack Brute Force': 'Web Attack'
})

# Ensure numeric features
X = df.drop(columns=['Label'])
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Max Normalization
for col in X.columns:
    m = abs(X[col].max())
    if m != 0:
        X[col] = X[col] / m

df = X.assign(Label = y)

# Robustly fill NaNs: Numeric with 0, Label with BENIGN
for col in df.columns:
    if col == 'Label':
        df[col] = df[col].fillna('BENIGN').astype(str)
    else:
        df[col] = df[col].fillna(0)

print("Preprocessing complete.")

## Separating Training and Testing db
We use the specific 70-30 split logic from the source code.

In [ ]:
df['is_train'] = np.random.uniform(0, 1, len(df)) <= .70
train, test = df[df['is_train']==True], df[df['is_train']==False]
print('Number of the training data:', len(train))
print('Number of the testing data:', len(test))

features = X.columns
y_train, label = pd.factorize(train['Label'])
y_test = test['Label'].map({name: i for i, name in enumerate(label)}).fillna(0).astype(int)

X_train = np.array(train[features])
X_test = np.array(test[features])
print("Training labels:", label)

## Model Construction
Hyperparameters: `n_estimators=50`, `learning_rate=1.0`.

In [ ]:
abc = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=0)
start = time.time()
model = abc.fit(X_train, y_train)
print(f"ELAPSE TIME MODEL: {(time.time() - start)/60:.2f} min")

## Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else model.predict(X_test)
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
print('Accuracy:', accuracy_score(y_test, y_pred))

unique_test_classes = np.unique(y_test.astype(str))
if len(unique_test_classes) > 1:
    indices = [i for i, name in enumerate(label) if name in unique_test_classes]
    y_test_bin = label_binarize(y_test, classes=label[indices].tolist())
    y_score_filtered = y_score[:, indices]
    try:
        print('ROC-AUC:', roc_auc_score(y_test_bin, y_score_filtered, multi_class='ovr'))
    except: print('ROC-AUC calculation failed')


## SHAP Analysis
Calculating SHAP values for 500 samples. This is model-agnostic and computationally intensive.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

start_index = 0
end_index = 100
test_data_for_shap = test[features].iloc[start_index:end_index]

print(f"Calculating SHAP values for {end_index} samples using KernelExplainer...")
explainer = shap.KernelExplainer(abc.predict_proba, test_data_for_shap)
shap_values = explainer.shap_values(test_data_for_shap)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, test_data_for_shap, class_names=label.tolist(), show=False, plot_type='bar')
plt.title("ADA Global Feature Importance (SHAP)")
plt.savefig('ADA_Shap_Summary_global.png', bbox_inches='tight')
plt.show()